# Utility Operations Intelligence — One-Shot Setup

This master notebook runs every automatable step end-to-end:

| Step | Action | Runtime |
| --- | --- | --- |
| 1 | Generate 13 source tables (~11.5M rows) | 15-25 min |
| 1b | Sanity SQL validation | 1 min |
| 2 | Build 2 rollup tables + 4 metric views | 2-3 min |
| 2b | Validate all 55 metric view measures | 2 min |
| 3 | Build 3 Genie Space create-payloads + create them | 1 min |
| 4 | Pull narrative anchors + upload pre-generated PDFs to UC Volume | 2-5 min |

**Manual steps NOT covered here** (do them in the workspace UI after this notebook completes):
- **Step 5** — Build the supervisor Multi-Agent System in **AI/ML → Agents → Create → Multi-Agent System**. See `5-supervisor-model/README.md`.
- **Step 6** — Deploy the Databricks App. See `6-app/README.md` or run `bash 6-app/deploy.sh` from your terminal.

---

## How to use

1. Import this whole repo into your Databricks workspace (Git Folder or upload).
2. Open `setup_all.ipynb`.
3. Set the widget values at the top (catalog, schema, warehouse, etc.).
4. **Run All**.
5. When it finishes, follow the manual steps 5 and 6 to build the supervisor and deploy the app.

Re-running is safe — every step is idempotent (`CREATE OR REPLACE`, `mode=overwrite`, etc.).


## ⚙️ Configuration

Set these parameters before running. They're widgets so they're easy to re-set across runs.

In [ ]:
# Define widgets the user can fill in.
# These get passed down to every child notebook via the global namespace.
dbutils.widgets.text("catalog", "main", "1. Catalog name")
dbutils.widgets.text("schema", "utility_ops_demo", "2. Schema name (will be created)")
dbutils.widgets.text("volume_name", "reports", "3. Volume name for PDFs")
dbutils.widgets.text("warehouse_id", "", "4. SQL Warehouse ID")
dbutils.widgets.text("user_email", "", "5. Your email (for Genie space parent path)")
dbutils.widgets.text("profile", "DEFAULT", "6. CLI profile (only used outside Databricks)")
dbutils.widgets.dropdown("skip_data_generation", "false", ["false", "true"], "7. Skip 11M-row data gen?")
dbutils.widgets.dropdown("skip_pdf_upload", "false", ["false", "true"], "8. Skip PDF upload?")


In [ ]:
# Read widgets into the global namespace.
# Child notebooks (run via `%run`) inherit these via globals().get("CATALOG", ...) fallbacks.
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME_NAME = dbutils.widgets.get("volume_name")
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
USER_EMAIL = dbutils.widgets.get("user_email")
PROFILE = dbutils.widgets.get("profile")
SKIP_DATA = dbutils.widgets.get("skip_data_generation") == "true"
SKIP_PDFS = dbutils.widgets.get("skip_pdf_upload") == "true"

assert CATALOG and SCHEMA, "catalog and schema are required"
assert WAREHOUSE_ID, "warehouse_id is required (find one in SQL → Warehouses)"
assert USER_EMAIL and "@" in USER_EMAIL, "user_email is required for Genie space parent_path"

print(f"Target: {CATALOG}.{SCHEMA}")
print(f"Volume: /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}")
print(f"Warehouse: {WAREHOUSE_ID}")
print(f"User: {USER_EMAIL}")
print(f"Skip data generation: {SKIP_DATA}")
print(f"Skip PDF upload: {SKIP_PDFS}")


In [ ]:
# Ensure the target catalog/schema/volume exist before any step writes to them.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME_NAME}")
print(f"✓ Schema {CATALOG}.{SCHEMA} ready")
print(f"✓ Volume /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME} ready")


---

## Step 1 — Generate source data

Creates the 13 source tables: `regions`, `sites`, `assets`, `technicians`, `market_prices`, `telemetry` (~10.5M rows), `sensor_anomalies`, `outages`, `maintenance_schedule`, `work_orders`, `asset_financials`, `power_sales`, `safety_incidents`.

**Runtime: 15-25 minutes.** Toggle `skip_data_generation` to `true` to skip if the tables already exist.

In [ ]:
import time
t0 = time.time()
if SKIP_DATA:
    print("⏭  Skipping data generation (widget set)")
else:
    print(f"⏳  Generating data in {CATALOG}.{SCHEMA} — this takes 15-25 minutes")


In [ ]:
%run ./1-data/generate

In [ ]:
if not SKIP_DATA:
    print(f"✓ Step 1 complete in {(time.time()-t0)/60:.1f} min")


### Step 1b — Sanity SQL

12 read-only queries that verify causal chains are intact (anomalies → outages, asset age vs financials, market price seasonality, etc.).

In [ ]:
%run ./1-data/sanity_sql

---

## Step 2 — Build metric views

Creates `asset_daily_summary` (~237k rows) + `region_daily_summary` (~5.8k rows) rollup tables, then 4 Unity Catalog Metric Views on top of them.

In [ ]:
%run ./2-metric-views/build_metric_views

### Step 2b — Validate every measure

Runs all 55 measures with multiple grouping dimensions. Expected: 52 OK, 3 expected zeros, 0 errors.

In [ ]:
%run ./2-metric-views/test_metric_views

---

## Step 3 — Build Genie Spaces

The build script writes 3 `serialized_space` + 3 `create-payload` JSON files to `/tmp/`. We then POST each one to the Genie API directly from this notebook.

In [ ]:
%run ./3-genie-spaces/build_genie_spaces

In [ ]:
# POST each create-payload to the Genie API and capture the resulting space_id.
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
created_spaces = {}

for name, path in [
    ("Grid Operations", "/tmp/grid_create.json"),
    ("Energy Financial Performance", "/tmp/fin_create.json"),
    ("Maintenance and Workforce Operations", "/tmp/maint_create.json"),
]:
    payload = json.loads(open(path).read())
    try:
        resp = w.api_client.do("POST", "/api/2.0/genie/spaces", body=payload)
        sid = resp.get("space_id")
        created_spaces[name] = sid
        print(f"✓ {name}: space_id={sid}")
    except Exception as e:
        # If the spaces already exist, this fails. Surface the error but continue.
        print(f"⚠  {name}: {e}")

print("\nGenie spaces:")
for name, sid in created_spaces.items():
    host = spark.conf.get("spark.databricks.workspaceUrl", "")
    print(f"  {name}: https://{host}/genie/rooms/{sid}" if host else f"  {name}: {sid}")


---

## Step 4 — Documents (PDFs to Volume)

The 50 PDFs in `4-documents/pdfs/` are pre-generated and ready to upload. We also refresh the narrative anchors (used if you later regenerate the PDFs).

In [ ]:
%run ./4-documents/anchor_narrative

In [ ]:
# Upload the 50 pre-generated PDFs to /Volumes/<catalog>/<schema>/<volume>/
import os
from pathlib import Path

if SKIP_PDFS:
    print("⏭  Skipping PDF upload (widget set)")
else:
    pdf_dir = Path(os.getcwd()) / "4-documents" / "pdfs"
    # Resolve relative to this notebook's location (when run from /Workspace/Repos/... pdf_dir might differ)
    if not pdf_dir.exists():
        # Try the notebook's parent directory
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        repo_root = Path(nb_path).parent
        pdf_dir = Path("/Workspace") / str(repo_root).lstrip("/") / "4-documents" / "pdfs"

    target_dir = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
    pdfs = sorted(Path(pdf_dir).glob("*.pdf")) if Path(pdf_dir).exists() else []
    print(f"Found {len(pdfs)} PDFs in {pdf_dir}")
    print(f"Uploading to {target_dir}/")

    for pdf in pdfs:
        dest = f"{target_dir}/{pdf.name}"
        dbutils.fs.cp(f"file:{pdf}", dest)

    print(f"✓ Uploaded {len(pdfs)} PDFs")
    print(f"  Listing first 5:")
    for f in dbutils.fs.ls(target_dir)[:5]:
        print(f"    {f.name} ({f.size} bytes)")


---

## ✅ Done — what's next

You now have:
- **13 source tables** in `{CATALOG}.{SCHEMA}` (run `SELECT * FROM {CATALOG}.{SCHEMA}.regions` to verify)
- **2 rollup tables** (`asset_daily_summary`, `region_daily_summary`)
- **4 metric views** (`grid_operations_metrics`, `financial_performance_metrics`, `maintenance_workforce_metrics`, `executive_summary_metrics`)
- **3 Genie Spaces** (URLs printed in Step 3)
- **50 PDFs** in `/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/`

### Remaining manual steps

**Step 5 — Build the supervisor agent (UI, ~10 min)**

1. AI/ML → Agents → Create → **Knowledge Assistant** over the PDFs volume.
2. AI/ML → Agents → Create → **Multi-Agent System** with all 3 Genie spaces + the Knowledge Assistant as tools.
3. Set supervisor LLM to **Claude Haiku 4.5**, enable parallel tool calls, enable streaming.

See `5-supervisor-model/README.md` for the system prompt, tool config, and performance tuning checklist.

**Step 6 — Deploy the app (from your terminal)**

```bash
cd 6-app/
bash deploy.sh
```

Update `app.yaml`'s `SUPERVISOR_ENDPOINT` value to the `mas-*` endpoint name from step 5 before deploying.
